## Experiment with P-adic Regularizer Formulation

### November 4, 2026

### Mukundan Thanigaivelan

Here, we work with a sample weight matrix to see how we can define a concrete ultrametric penalty.

### Load libraries

In [1]:
import pandas as pd
import tensorflow as tf
import h5py
import os
from zipfile import ZipFile

I0000 00:00:1793848208.092021 4125954 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793848208.221815 4125954 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793848228.780657 4125954 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/scratch/user/u.mt372094/conda/envs/uadage-conda/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).


### Experimentation

In [2]:
# Pull a weight matrix
with ZipFile("../data/ec_unreg_model_1.keras") as z:
  z.extract('model.weights.h5')

data = {}
with h5py.File('model.weights.h5', 'r') as f:
  def extract(name, obj):
    if isinstance(obj, h5py.Dataset):
      data[name] = obj[()]
  f.visititems(extract)

os.remove('model.weights.h5')

weights = pd.DataFrame(data[list(data.keys())[0]])
weights.shape

(3608, 70)

In [15]:
weights.head()

,0,1,2,3,4,5,6,7,8,9,...,60,61,62,63,64,65,66,67,68,69
0,-0.017321,-0.034606,0.016899,0.033786,-0.013438,-0.045978,-0.005880,0.045779,-0.018970,0.001801,...,0.026325,0.035501,0.013882,0.030619,0.000226,-0.013005,0.008689,0.005030,-0.016102,-0.004255
1,0.052765,-0.012167,-0.022767,0.019053,0.026636,0.001241,-0.008806,0.024628,-0.039547,0.013808,...,-0.006102,0.007526,-0.005409,-0.011427,-0.009697,0.014987,-0.010922,0.000876,0.026478,0.016311
2,0.003396,-0.011480,-0.000146,-0.035550,0.011078,-0.048387,0.009265,0.025116,0.023974,0.003853,...,-0.030943,0.009529,0.008772,0.013278,0.020838,0.000854,-0.009887,-0.012748,0.003276,0.002790
3,0.000403,0.007207,0.042067,0.006554,-0.029734,-0.010457,-0.016389,0.026284,0.036070,-0.017969,...,0.037775,0.008795,-0.018102,0.024625,-0.032550,-0.023644,-0.009867,0.002363,0.009948,-0.001307
4,0.007469,0.000091,-0.014340,0.012488,0.018689,-0.006670,-0.014093,-0.027601,-0.008263,0.049399,...,-0.005943,0.020991,0.041628,-0.011448,-0.015349,0.002783,0.011423,-0.007056,-0.001555,0.045804


In [19]:
# Transpose to get nodes x genes
nodes = tf.transpose(weights)
num_nodes = tf.shape(nodes)[0]
num_nodes

<tf.Tensor: shape=(), dtype=int32, numpy=70>

In [21]:
# Create a random noise matrix of shape (256, num_nodes)
num_triplets = 256
shape_tensor = tf.stack([num_triplets, num_nodes])
random_scores = tf.random.uniform(shape_tensor)

In [24]:
# Get array of triplets where each triplet has different values
_, unique_indices = tf.math.top_k(random_scores, k=3)
unique_indices[:10]

<tf.Tensor: shape=(10, 3), dtype=int32, numpy=
array([[17, 58,  2],
       [14, 63, 45],
       [12, 29,  3],
       [53, 61, 47],
       [44, 21, 43],
       [ 3, 28,  5],
       [46, 62, 56],
       [46, 36, 54],
       [16, 36, 47],
       [14, 18,  2]], dtype=int32)>

In [27]:
# Generates 256 random indices between 0 and num_nodes (repeats thrice for A, B, and C in calculation)
idx_A = unique_indices[:, 0]
idx_B = unique_indices[:, 1]
idx_C = unique_indices[:, 2]

In [28]:
# Tensor --> array of 256 values between 0 and num_nodes
idx_A

<tf.Tensor: shape=(256,), dtype=int32, numpy=
array([17, 14, 12, 53, 44,  3, 46, 46, 16, 14, 46, 68, 26, 57, 41, 29,  4,
       42, 55, 16, 44, 45, 60, 41, 67, 17, 23, 66, 20, 23, 42, 26, 22, 30,
       68, 65, 54, 64, 57, 13,  5, 41, 56, 10, 67, 14, 45, 19, 66, 60, 10,
       24, 49, 35, 44, 25, 44, 69, 65, 47, 13, 39, 67, 55, 18, 28, 25, 46,
       57, 60,  9, 65, 62, 20, 38, 65, 16, 55, 66,  4, 23,  2, 35,  2, 45,
       53, 32, 20, 20, 23, 20,  2, 10, 42, 53, 14, 30, 69, 68, 43, 44, 21,
       30, 65, 48, 60, 60, 35, 54,  8, 37, 17, 34, 64, 12, 50, 51,  4,  1,
       27,  6,  5, 14, 26, 57, 65, 17, 41, 18, 42, 12, 60, 41, 42, 55, 37,
       29, 22, 56,  3,  5, 34, 21, 56, 35, 27,  5,  2, 23, 65, 23,  8, 64,
        8, 56,  1, 28,  9, 36, 39, 15, 36, 68, 54, 66,  9, 56, 48, 18, 67,
       23, 64, 38, 56, 21,  6, 51, 50, 66, 57, 35,  4, 63, 63, 21, 55,  7,
        8, 17, 42, 41, 49, 35, 69, 39, 28, 64,  9, 53, 24, 42, 41, 19,  2,
       59, 61, 22, 42,  9, 48, 56, 37, 22, 37, 11, 59,

In [30]:
# Pull 256 x 3 = 768 nodes as 256 triplets
A = tf.gather(nodes, idx_A)
B = tf.gather(nodes, idx_B)
C = tf.gather(nodes, idx_C)
A

<tf.Tensor: shape=(256, 3608), dtype=float32, numpy=
array([[-1.82403456e-02, -8.02253932e-03, -1.75748225e-02, ...,
        -1.70306563e-02,  6.58806833e-03, -3.88872065e-02],
       [-1.06799481e-02,  9.31242667e-03, -1.97665058e-02, ...,
         1.38449911e-02,  3.50142121e-02, -5.00001406e-05],
       [-1.69025362e-02,  2.15037577e-02, -2.27627670e-03, ...,
        -3.37226950e-02, -2.35393327e-02, -4.46485579e-02],
       ...,
       [ 3.30462074e-03,  5.42241614e-03,  1.17176045e-02, ...,
         1.68428998e-02,  1.85419573e-04,  1.05436854e-02],
       [ 1.94622092e-02,  4.05836571e-03,  2.01525986e-02, ...,
         1.90902073e-02,  2.17327047e-02, -3.48169580e-02],
       [-1.82403456e-02, -8.02253932e-03, -1.75748225e-02, ...,
        -1.70306563e-02,  6.58806833e-03, -3.88872065e-02]],
      shape=(256, 3608), dtype=float32)>

In [32]:
# Compute pairwise distances --> each one is a vector of 256 distances
d_AB = tf.sqrt(tf.reduce_sum(tf.square(A - B), axis=1) + 1e-8)
d_BC = tf.sqrt(tf.reduce_sum(tf.square(B - C), axis=1) + 1e-8)
d_AC = tf.sqrt(tf.reduce_sum(tf.square(A - C), axis=1) + 1e-8)
d_AB.shape

TensorShape([256])

In [33]:
# Sort distances in descending order
distances = tf.stack([d_AB, d_BC, d_AC], axis=1)
sorted_distances = tf.sort(distances, axis=1, direction='DESCENDING')

In [36]:
# Get longest and 2nd-longest sides
d1 = sorted_distances[:, 0] # Longest side
d2 = sorted_distances[:, 1]
d1.shape

TensorShape([256])

In [39]:
# Calculate penalty
lambda_ultra = 0.5
penalty = tf.reduce_mean(tf.square(d1 - d2))
float(lambda_ultra * penalty)

0.0002002635010285303